# IBM HR Analytics — Employee Attrition EDA

**Track I1 — EDA & Visualization**

**Dataset:** IBM HR Analytics Employee Attrition & Performance (Kaggle, released by
IBM as a synthetic/sample dataset for HR analytics demonstrations — not real
employee records).

**Goal:** Understand what patterns in this dataset are associated with employee
attrition, using structured univariate and bivariate exploratory analysis.

## 1. Data Understanding

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

df = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())

Shape: 1470 rows, 35 columns
   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   EnvironmentSatisfaction  Gender  HourlyRat

### 1.1 Data types, missing values, duplicates, and constant columns

Before any analysis, we check the structural quality of the data:
- **Data types** — are numeric columns actually numeric, categorical columns stored as text?
- **Missing values** — any nulls that need imputing or flagging?
- **Duplicate rows** — any exact repeats that would double-count observations?
- **Constant columns** — columns with only one unique value carry zero information and should be dropped.
- **ID integrity** — does `EmployeeNumber` uniquely identify each row, as an ID should?

In [4]:
print("--- Data types ---")
print(df.dtypes.value_counts())
print()

print("--- Missing values ---")
missing = df.isnull().sum()
print(f"Total missing values across all columns: {missing.sum()}")
print()

print("--- Duplicate rows ---")
print(f"Duplicate rows: {df.duplicated().sum()}")
print()

print("--- Columns with only 1 unique value (no information) ---")
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
for c in constant_cols:
    print(f"  {c}: constant value = {df[c].unique()[0]}")
print()

print("--- EmployeeNumber uniqueness (should behave like an ID) ---")
print(f"Unique EmployeeNumber values: {df['EmployeeNumber'].nunique()} out of {len(df)} rows")
print()

print("--- Object (text) columns and their unique value counts ---")
obj_cols = df.select_dtypes(include=["object", "string"]).columns
for c in obj_cols:
    print(f"  {c}: {df[c].nunique()} unique values -> {list(df[c].unique())[:6]}")

--- Data types ---
int64    26
str       9
Name: count, dtype: int64

--- Missing values ---
Total missing values across all columns: 0

--- Duplicate rows ---
Duplicate rows: 0

--- Columns with only 1 unique value (no information) ---
  EmployeeCount: constant value = 1
  Over18: constant value = Y
  StandardHours: constant value = 80

--- EmployeeNumber uniqueness (should behave like an ID) ---
Unique EmployeeNumber values: 1470 out of 1470 rows

--- Object (text) columns and their unique value counts ---
  Attrition: 2 unique values -> ['Yes', 'No']
  BusinessTravel: 3 unique values -> ['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
  Department: 3 unique values -> ['Sales', 'Research & Development', 'Human Resources']
  EducationField: 6 unique values -> ['Life Sciences', 'Other', 'Medical', 'Marketing', 'Technical Degree', 'Human Resources']
  Gender: 2 unique values -> ['Female', 'Male']
  JobRole: 9 unique values -> ['Sales Executive', 'Research Scientist', 'Laboratory Tec

### 1.2 Cleaning decisions

Based on the checks above:
- Drop `EmployeeCount`, `Over18`, `StandardHours` — constant columns, zero information.
- Keep `EmployeeNumber` as an identifier only; excluded from statistical analysis.
- No missing values or duplicates to handle.

All further analysis uses `df_clean`, the cleaned working dataframe.

In [5]:
cols_to_drop = ["EmployeeCount", "Over18", "StandardHours"]
df_clean = df.drop(columns=cols_to_drop)

print(f"Dropped columns: {cols_to_drop}")
print(f"Shape before: {df.shape} -> Shape after: {df_clean.shape}")
print()
print("Remaining columns:")
print(list(df_clean.columns))

Dropped columns: ['EmployeeCount', 'Over18', 'StandardHours']
Shape before: (1470, 35) -> Shape after: (1470, 32)

Remaining columns:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']
